# M4 Validation: Delta Walk Traversal
End-to-end M4 validation with A1 (VAE encode/decode) and A3 latent injection.

In [ ]:
import time
import torch
from pathlib import Path
from IPython.display import display
from helpers.methods.access import encode_image_to_latents, decode_latents_to_image
from helpers.methods.navigation import delta_walks
from helpers.runtime.pipeline import default_torch_dtype_for_device, load_flux_pipeline, resolve_runtime_device
from helpers.runtime.sampling import generate_with_prompt
from helpers.runtime.utility import flush_namespace

In [ ]:
output_dir = Path('../python/examples/navigation/outputs/m4_delta_walk_notebook').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

device = resolve_runtime_device()
dtype = default_torch_dtype_for_device(torch.float16, device)
try:
    pipe = load_flux_pipeline(
        torch_dtype=dtype,
        device=device,
        offload_mode="group",
        group_use_stream=True,
        group_low_cpu_mem_usage=False,
        enable_text_encoder_group_offload=True,
        text_encoder_group_offload_type="block_level",
        text_encoder_num_blocks_per_group=1,
        enable_vae_group_offload=True,
        vae_group_offload_type="leaf_level",
        vae_num_blocks_per_group=2,
        enable_vae_slicing=True,
        enable_vae_tiling=True,
        enable_layerwise_casting=True,
        layerwise_storage_dtype="float8_e4m3fn",
        layerwise_compute_dtype=dtype,
        enable_channels_last=True,
        enable_xformers_attention=True,
        enable_torch_compile=False,
    )

    prompt = 'a calm coastal village with detailed architecture'
    with torch.inference_mode():
        anchor_a = generate_with_prompt(pipe=pipe, prompt=prompt, seed=12, num_inference_steps=4, guidance_scale=0.0).images[0]
        anchor_b = generate_with_prompt(pipe=pipe, prompt=prompt, seed=96, num_inference_steps=4, guidance_scale=0.0).images[0]
    display(anchor_a)
    display(anchor_b)

    latent_a = encode_image_to_latents(pipe=pipe, image=anchor_a, size=None, sample_posterior=False, device='cpu', dtype=dtype)
    latent_b = encode_image_to_latents(pipe=pipe, image=anchor_b, size=None, sample_posterior=False, device='cpu', dtype=dtype)
    preview = decode_latents_to_image(pipe, latent_a)
    display(preview)

    for scale, walked in zip([0.0, 0.5, 1.0, 1.5], delta_walks(latent_a, latent_b, [0.0, 0.5, 1.0, 1.5])):
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = generate_with_prompt(
                pipe=pipe,
                prompt=prompt,
                seed=321,
                latents=walked,
                num_inference_steps=4,
                guidance_scale=0.0,
            )
        t1 = time.perf_counter()
        image = out.images[0]
        
        path = output_dir / f'm4_delta_scale_{scale:+.2f}_{t1 - t0:.2f}s.png'
        image.save(path)
        
        print(f'scale={scale:+.2f} in {t1 - t0:.2f}s -> {path}')
        display(image)
finally:
    flush_namespace(
        globals(),
        "pipe",
        "out",
        "image",
        "anchor_a",
        "anchor_b",
        "latent_a",
        "latent_b",
        "preview",
        verbose=True,
    )